In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==0.4.6
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
       bi2012(),
       #BI2013a(),
       #BI2014a(),
       #BI2014b(),
       #BI2015a(),
       #BI2015b(),
       BNCI2014008(),
       BNCI2014009(),
       #BNCI2015_003(),
       #Cattan2019_VR(),
       #EPFLP300(),
       #Huebner2017(),
       #Huebner2018(),
       #Lee2019_ERP(),
       #Sosulski2019()   
   ]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda",
    overwrite=False,
    random_state=42,
    n_jobs=5,
    #data_size=dict(
    #    policy='per_class',
    #    value=[10]
    #),
    #n_perms=[2],
)

In [3]:
from sklearn.pipeline import make_pipeline, Pipeline
from mne.decoding import Scaler
from hoda.hoda import HODA, BTTDA
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from pyriemann.estimation import XdawnCovariances
from pyriemann.tangentspace import TangentSpace
from hoda.classification import ToeplitzLDAWrapper, Vectorize
from sklearn.feature_selection import SelectFwe, SelectKBest, RFECV
from sklearn.model_selection import GridSearchCV


pipelines = dict()


pipelines['tLDA'] = ToeplitzLDAWrapper()
"""
pipelines['HODA_cv'] = GridSearchCV(
        Pipeline([
            ('hoda', HODA(
                max_iter=64,
                tol=1e-6,
                init ='svd',
                shrinkage='lw',
                toeplitz=None,
                obj='rt',
                solver='lanczos',
                taper=False,
                keep_train_info=False,
                verbose=False,
                prune=False,
            )),
            ('vec', Vectorize()),
            ('select', SelectFwe()),
            ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
        ]),
        dict(hoda__rank=[[k,k] for k in range(1,8+1)]),
        scoring='roc_auc',
)
pipelines['HODA_prune'] = Pipeline([
    ('hoda', HODA(
        max_iter=1024,
        tol=1e-6,
        init ='svd',
        shrinkage='lw',
        toeplitz=None,
        obj='rt',
        solver='lanczos',
        taper=False,
        keep_train_info=False,
        verbose=False,
        prune=True,
    )),
    ('vec', Vectorize()),
    ('select', SelectFwe()),
    ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])
"""
pipelines['PARAFAC-DA'] = Pipeline([
    ('bttda', BTTDA(
        n_blocks=16,
        hoda_params=dict(
            rank=[1,1],
            max_iter=256,
            tol=1e-6,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='tr',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
            prune=True,
        ),
        info_crit='bic',
        verbose=False,
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])
pipelines['PARAFAC-DA_16'] = Pipeline([
    ('bttda', BTTDA(
        n_blocks=16,
        hoda_params=dict(
            rank=[1,1],
            max_iter=256,
            tol=1e-6,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='tr',
            solver='lanczos',
            taper=False,
            keep_train_info=False,
            verbose=False,
            prune=True,
        ),
        info_crit=None,
        verbose=False,
    )),
    ('vec', Vectorize()),
    ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

"""
pipelines['sLDA'] = make_pipeline(
        Vectorize(),
        LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)


pipelines['XDAWN+RG'] = make_pipeline(
    XdawnCovariances(n_components=3),
    TangentSpace(metric="riemann"),
    LogisticRegression(),
)
"""

'\npipelines[\'sLDA\'] = make_pipeline(\n        Vectorize(),\n        LinearDiscriminantAnalysis(shrinkage=\'auto\', solver=\'lsqr\')\n)\n\n\npipelines[\'XDAWN+RG\'] = make_pipeline(\n    XdawnCovariances(n_components=3),\n    TangentSpace(metric="riemann"),\n    LogisticRegression(),\n)\n'

In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

Brain Invaders 2012-WithinSession:   0%| | 0/25 [00:00<?, 

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Brain Invaders 2012-WithinSession:   4%| | 1/25 [00:44<17:

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Brain Invaders 2012-WithinSession:   8%| | 2/25 [01:14<13:

...

/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Brain Invaders 2012-WithinSession:  44%|▍| 11/25 [07:53<10

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())
Brain Invaders 2012-WithinSession:  48%|▍| 12/25 [08:27<08

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())


In [ ]:
results=results[results['dataset']!='EPFL P300 dataset']
results

In [ ]:
import seaborn as sns
order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()

sns.barplot(
     data=results,
    y="score", x="dataset", hue="pipeline", hue_order=order.index,
)

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'tLDA', 'PARAFAC-DA')
_  = paired_plot(results, 'tLDA', 'PARAFAC-DA')

In [ ]:
results

In [ ]:
df = results.pivot_table(index=['dataset', 'subject', 'session'],columns='pipeline', values='score')
ax = sns.scatterplot(data=df, x='HODA_prune', y='HODA_prune_cv', style='dataset', hue='subject', palette='Set1')
ax.plot([.5,1],[.5,1], color='black', )